In [1]:
import os
import math
import numpy as np
import xarray as xr
import scipy.ndimage
from pathlib import Path
import netCDF4 as nc
import scipy.linalg
import time
import sys
import multiprocessing
import json
import gc
import cc3d
import argparse

ql_threshold = 10**-5
shell_prop_lower_threshold = 0.2

def label(source):
    #source must be (z,y,x)
    #returns a labeled version wrapping across x,y

    padded_source = np.pad(source, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)
    padded_labeled_source = cc3d.connected_components(padded_source, connectivity=6, periodic_boundary=True)
    return padded_labeled_source[1:-1, :, :]

def periodic_pad(original):
    # pad
    padded = np.pad(original, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=0)

    # wrap x and y
    padded[:, 0, :] = padded[:, -2, :]
    padded[:, -1, :] = padded[:, 1, :]
    padded[:, :, 0] = padded[:, :, -2]
    padded[:, :, -1] = padded[:, :, 1]

    return cc3d.connected_components(
        padded.astype(np.uint32), 
        connectivity=6, 
        periodic_boundary=True,
        binary_image=True
    )

def strip_small_components(original):
    # ----- Input -----
    # original : numpy to be stripped of small components (must be z,y,x)
    # ----- Output -----
    # stripped_labels
    # stripped_mask

    if not np.any(original):
        return np.zeros_like(original, dtype=np.int32), np.zeros_like(original, dtype=bool)

    # pad
    padded = np.pad(original, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=0)

    # wrap x and y
    padded[:, 0, :] = padded[:, -2, :]
    padded[:, -1, :] = padded[:, 1, :]
    padded[:, :, 0] = padded[:, :, -2]
    padded[:, :, -1] = padded[:, :, 1]

    # Cast to uint32 and use binary_image=True to guarantee cc3d processes this as binary foreground
    padded_init_labels = cc3d.connected_components(
        padded.astype(np.uint32), 
        connectivity=6, 
        periodic_boundary=True,
        binary_image=True
    )

    thick_core_footprint = np.ones((2, 2, 2), dtype=bool)
    binary_init_labels = (padded_init_labels > 0)
    eroded = scipy.ndimage.binary_erosion(binary_init_labels, structure=thick_core_footprint)

    surviving_ids = np.unique(padded_init_labels[eroded])
    valid_labels = surviving_ids[surviving_ids != 0]

    if len(valid_labels) == 0:
        # If the 2x2x2 erosion was too aggressive and wiped everything out:
        return np.zeros_like(original, dtype=np.int32), np.zeros_like(original, dtype=bool)
    
    init_labels = padded_init_labels[1:-1, 1:-1, 1:-1]
    stripped_mask = np.isin(init_labels, valid_labels)

    stripped_labels = label(stripped_mask)

    return stripped_labels, stripped_mask

In [2]:
source_input_dir = Path("/mnt/stor-pool-01/projects/heus/BNF/Ensemble_Tests/512_512_50m_12km_chbuff/2025-07-01_normal_shell/thijs")

paths = {
        "ql": source_input_dir / "ql.nc",
        "shell": source_input_dir / "shell.nc"
    }

In [3]:
t=154800

#load datasets
with xr.open_dataset(paths["ql"], decode_times=False, engine="netcdf4") as ds_ql, \
    xr.open_dataset(paths["shell"], decode_times=False, engine="netcdf4") as ds_shell_prop:

    ql_raw = (ds_ql.ql.sel(time=t).fillna(0) > ql_threshold).compute().values.astype(bool)
    shell_prop_raw = ds_shell_prop.sel(time=t)
    shell_prop_mask = ((shell_prop_raw.shell >= shell_prop_lower_threshold) & (shell_prop_raw.shell < 1)).compute().values.astype(bool)
    cloud_prop_mask = (shell_prop_raw.shell > 0.99).compute().values.astype(bool)
    
        
#--- Step 1 : Obtain Filtered Cloud Mask ---
#strip small parts of ql
ql_labels, ql_mask = strip_small_components(ql_raw)

#apply to cloud mask
restricted_cloud_mask = ql_mask & cloud_prop_mask

local_cloud_mask = restricted_cloud_mask
local_cloud_labels = label(local_cloud_mask)

#--- Step 2: Obtain Shell Labels ---
#make labels for combined object

local_shell_mask = shell_prop_mask & ~local_cloud_mask

local_combined_mask = local_shell_mask | local_cloud_mask
local_combined_labels = label(local_combined_mask)

#strip to just the shell
local_shell_labels = np.where(local_shell_mask, local_combined_labels, 0)

#--- Step 3: Find Free Standing Shells ---
#get cloud labels for combined objects
connected_labels = np.where(local_cloud_mask, local_combined_labels, 0)

#turn the labels into a list
connected_ids = np.unique(connected_labels) #Don't drop background because we are negating this next

#obtain free shell mask
local_free_shell_mask = ~np.isin(local_shell_labels, connected_ids)

#mask the labels
local_free_shell_labels = np.where(local_free_shell_mask, local_shell_labels, 0)

In [4]:
np.unique(ql_labels)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [5]:
np.unique(local_cloud_labels)

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

In [6]:
np.unique(restricted_cloud_mask)

array([False,  True])

In [9]:
ds_ql = xr.open_dataset(paths["ql"], decode_times=False, engine="netcdf4")


In [10]:
ds_ql.time.values

array([     0.,   3500.,   3600.,   7100.,   7200.,  10700.,  10800.,
        14300.,  14400.,  17900.,  18000.,  21500.,  21600.,  25100.,
        25200.,  28700.,  28800.,  32300.,  32400.,  35900.,  36000.,
        39500.,  39600.,  43100.,  43200.,  46700.,  46800.,  50300.,
        50400.,  53900.,  54000.,  57500.,  57600.,  61100.,  61200.,
        64700.,  64800.,  68300.,  68400.,  71900.,  72000.,  75500.,
        75600.,  79100.,  79200.,  82700.,  82800.,  86300.,  86400.,
        89900.,  90000.,  93500.,  93600.,  97100.,  97200., 100700.,
       100800., 104300., 104400., 107900., 108000., 111500., 111600.,
       115100., 115200., 118700., 118800., 122300., 122400., 125900.,
       126000., 129500., 129600., 133100., 133200., 136700., 136800.,
       140300., 140400., 143900., 144000., 147500., 147600., 151100.,
       151200., 154700., 154800., 158300., 158400., 161900., 162000.,
       165500., 165600., 169100., 169200., 172700., 172800., 176300.,
       176400., 1799

In [11]:
ds_shell_prop.time.values

array([     0.,   3500.,   3600.,   7100.,   7200.,  10700.,  10800.,
        14300.,  14400.,  17900.,  18000.,  21500.,  21600.,  25100.,
        25200.,  28700.,  28800.,  32300.,  32400.,  35900.,  36000.,
        39500.,  39600.,  43100.,  43200.,  46700.,  46800.,  50300.,
        50400.,  53900.,  54000.,  57500.,  57600.,  61100.,  61200.,
        64700.,  64800.,  68300.,  68400.,  71900.,  72000.,  75500.,
        75600.,  79100.,  79200.,  82700.,  82800.,  86300.,  86400.,
        89900.,  90000.,  93500.,  93600.,  97100.,  97200., 100700.,
       100800., 104300., 104400., 107900., 108000., 111500., 111600.,
       115100., 115200., 118700., 118800., 122300., 122400., 125900.,
       126000., 129500., 129600., 133100., 133200., 136700., 136800.,
       140300., 140400., 143900., 144000., 147500., 147600., 151100.,
       151200., 154700., 154800., 158300., 158400., 161900., 162000.,
       165500., 165600., 169100., 169200., 172700., 172800., 176300.,
       176400., 1799

In [16]:
ds_e = xr.open_dataset(source_input_dir / "netE.nc", decode_times=False, engine="netcdf4")


In [17]:
formatted_times = [f"{val:.6f}" for val in ds_e.netE_flux_y_shell.time.values]
print(formatted_times)

['3600.000000', '7200.000000', '10800.000000', '14400.000000', '18000.000000', '21600.000000', '25200.000000', '28800.000000', '32400.000000', '36000.000000', '39600.000000', '43200.000000', '46800.000000', '50400.000000', '54000.000000', '57600.000000', '61200.000000', '64800.000000', '68400.000000', '72000.000000', '75600.000000', '79200.000000', '82800.000000', '86400.000000', '90000.000000', '93600.000000', '97200.000000', '100800.000000', '104400.000000', '108000.000000', '111600.000000', '115200.000000', '118800.000000', '122400.000000', '126000.000000', '129600.000000', '133200.000000', '136800.000000', '140400.000000', '144000.000000', '147600.000000', '151200.000000', '154800.000000', '158400.000000', '162000.000000', '165600.000000', '169200.000000', '172800.000000', '176400.000000', '180000.000000', '183600.000000', '187200.000000', '190800.000000', '194400.000000', '198000.000000', '201600.000000', '205200.000000', '208800.000000', '212400.000000', '216000.000000', '219600.

In [18]:
ds_ql.time.values[84]

np.float32(151200.0)

In [20]:
# Testing time slicing

times = ds_e.netE_flux_y_shell.time.values
times_above = times[times > 154800]

start_time = 154800
exact_steps = [t for t in times_above if (t - start_time) % 7200 == 0]

np.set_printoptions(suppress=True)
print(np.array(exact_steps))

[162000. 169200. 176400. 183600. 190800. 198000. 205200. 212400. 219600.
 226800. 234000. 241200. 248400. 255600. 262800. 270000. 277200. 284400.
 291600. 298800. 306000. 313200. 320400. 327600. 334800. 342000. 349200.
 356400. 363600. 370800. 378000. 385200. 392400. 399600. 406800. 414000.
 421200. 428400. 435600. 442800. 450000. 457200. 464400. 471600. 478800.
 486000. 493200. 500400. 507600. 514800. 522000. 529200. 536400. 543600.
 550800. 558000. 565200. 572400. 579600.]


In [21]:
# Testing time slicing

times = ds_ql.time.values
times_above = times[times > 154800]

start_time = 154800
exact_steps = [t for t in times_above if (t - start_time) % 7200 == 0]

np.set_printoptions(suppress=True)
print(np.array(exact_steps))

[162000. 169200. 176400. 183600. 190800. 198000. 205200. 212400. 219600.
 226800. 234000. 241200. 248400. 255600. 262800. 270000. 277200. 284400.
 291600. 298800. 306000. 313200. 320400. 327600. 334800. 342000. 349200.
 356400. 363600. 370800. 378000. 385200. 392400. 399600. 406800. 414000.
 421200. 428400. 435600. 442800. 450000. 457200. 464400. 471600. 478800.
 486000. 493200. 500400. 507600. 514800. 522000. 529200. 536400. 543600.
 550800. 558000. 565200. 572400. 579600.]
